In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split

data = pd.read_csv('data/car_prices.csv')

print('Форма датасета:', data.shape)
data.head()

Форма датасета: (500, 11)


,make,fuel_type,body_style,drive_wheels,engine_size,horsepower,curb_weight,city_mpg,highway_mpg,year,price
0,Audi,gas,hardtop,awd,79.0,85.976330,2495.0,26.0,34.0,2022,35196.847075
1,Mercedes,diesel,hatchback,rwd,264.0,NaN,3109.0,35.0,42.0,2018,37780.794000
2,Hyundai,gas,sedan,fwd,187.0,195.935780,1856.0,31.0,36.0,2021,34280.199067
3,Ford,gas,hardtop,rwd,149.0,147.555812,1818.0,44.0,49.0,2021,20121.559279
4,Audi,diesel,wagon,fwd,162.0,150.196980,1912.0,NaN,38.0,2015,35371.154587


In [4]:
data.describe()

,engine_size,horsepower,curb_weight,city_mpg,highway_mpg,year,price
count,500.00000,470.000000,470.000000,470.000000,500.000000,500.000000,500.000000
mean,199.63400,158.787037,2899.027660,28.521277,34.908000,2016.106000,30480.672077
std,72.47002,62.072018,633.921505,9.257889,9.435125,3.778799,12250.114663
min,70.00000,50.000000,1807.000000,13.000000,16.000000,2010.000000,4000.000000
25%,142.00000,111.812983,2333.750000,21.000000,27.000000,2013.000000,22207.525372
50%,201.50000,158.197814,2894.500000,29.000000,35.000000,2016.000000,30215.294472
75%,255.50000,205.104533,3437.500000,37.000000,43.000000,2019.000000,38687.811968
max,329.00000,300.000000,3988.000000,44.000000,53.000000,2022.000000,63767.290061


In [5]:
data.dtypes

make                str
fuel_type           str
body_style          str
drive_wheels        str
engine_size     float64
horsepower      float64
curb_weight     float64
city_mpg        float64
highway_mpg     float64
year              int64
price           float64
dtype: object

In [ ]:
y = data['price']

X = data.drop(['price'], axis=1).select_dtypes(exclude=['object'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

print('Размер обучающей выборки:', X_train.shape)
print('Размер тестовой выборки:', X_test.shape)

Размер обучающей выборки: (400, 6)
Размер тестовой выборки: (100, 6)


In [7]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

def score_dataset(X_train, X_test, y_train, y_test):
    model = RandomForestRegressor(n_estimators=10, random_state=0)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    return mean_absolute_error(y_test, preds)

In [8]:
print('Размер тренировочной выборки:', X_train.shape)

missing_val_count = X_train.isnull().sum()
print('\nПропуски по столбцам:')
print(missing_val_count[missing_val_count > 0])

Размер тренировочной выборки: (400, 6)

Пропуски по столбцам:
horsepower     24
curb_weight    24
city_mpg       23
dtype: int64


In [9]:
cols_with_missing = [col for col in X_train.columns if X_train[col].isnull().any()]

reduced_X_train = X_train.drop(cols_with_missing, axis=1)
reduced_X_test = X_test.drop(cols_with_missing, axis=1)

print('MAE при удалении столбцов с пропусками:')
print(score_dataset(reduced_X_train, reduced_X_test, y_train, y_test))

MAE при удалении столбцов с пропусками:
8821.413953991296


In [10]:
from sklearn.impute import SimpleImputer

my_imputer = SimpleImputer()  # по умолчанию заполняет средним
imputed_X_train = pd.DataFrame(my_imputer.fit_transform(X_train))
imputed_X_test = pd.DataFrame(my_imputer.transform(X_test))

imputed_X_train.columns = X_train.columns
imputed_X_test.columns = X_test.columns

print('MAE при заполнении пропусков средним:')
print(score_dataset(imputed_X_train, imputed_X_test, y_train, y_test))

MAE при заполнении пропусков средним:
9277.540591827374


In [11]:
X_train_plus = X_train.copy()
X_test_plus = X_test.copy()

for col in cols_with_missing:
    X_train_plus[col + '_was_missing'] = X_train_plus[col].isnull()
    X_test_plus[col + '_was_missing'] = X_test_plus[col].isnull()

X_train_plus.head()

,engine_size,horsepower,curb_weight,city_mpg,highway_mpg,year,horsepower_was_missing,curb_weight_was_missing,city_mpg_was_missing
107,112.0,57.678973,2728.0,16.0,20.0,2016,False,False,False
336,325.0,224.054479,2450.0,33.0,41.0,2019,False,False,False
71,142.0,89.615050,2153.0,33.0,36.0,2015,False,False,False
474,119.0,92.691966,2112.0,27.0,31.0,2022,False,False,False
6,202.0,NaN,2378.0,36.0,43.0,2021,True,False,False


In [12]:
my_imputer = SimpleImputer()
imputed_X_train_plus = pd.DataFrame(my_imputer.fit_transform(X_train_plus))
imputed_X_test_plus = pd.DataFrame(my_imputer.transform(X_test_plus))

imputed_X_train_plus.columns = X_train_plus.columns
imputed_X_test_plus.columns = X_test_plus.columns

print('MAE при заполнении пропусков + флаг отсутствия:')
print(score_dataset(imputed_X_train_plus, imputed_X_test_plus, y_train, y_test))

MAE при заполнении пропусков + флаг отсутствия:
9269.753508757054


In [ ]:
y = data['price']
X_full = data.drop(['price'], axis=1)

X_train_full, X_test_full, y_train, y_test = train_test_split(
    X_full, y, train_size=0.8, test_size=0.2, random_state=0
)

cols_with_missing = [col for col in X_train_full.columns if X_train_full[col].isnull().any()]
X_train_full.drop(cols_with_missing, axis=1, inplace=True)
X_test_full.drop(cols_with_missing, axis=1, inplace=True)

low_cardinality_cols = [
    cname for cname in X_train_full.columns
    if X_train_full[cname].nunique() < 15 and X_train_full[cname].dtype == 'object'
]
numerical_cols = [
    cname for cname in X_train_full.columns
    if X_train_full[cname].dtype in ['int64', 'float64']
]

print('Категориальные признаки:', low_cardinality_cols)
print('Числовые признаки:', numerical_cols)

my_cols = low_cardinality_cols + numerical_cols
X_train = X_train_full[my_cols].copy()
X_test = X_test_full[my_cols].copy()

X_train.head()

Категориальные признаки: []
Числовые признаки: ['engine_size', 'highway_mpg', 'year']


,engine_size,highway_mpg,year
107,112.0,20.0,2016
336,325.0,41.0,2019
71,142.0,36.0,2015
474,119.0,31.0,2022
6,202.0,43.0,2021


In [14]:
drop_X_train = X_train.select_dtypes(exclude=['object'])
drop_X_test = X_test.select_dtypes(exclude=['object'])

print('MAE без категориальных признаков:')
print(score_dataset(drop_X_train, drop_X_test, y_train, y_test))

MAE без категориальных признаков:
8821.413953991296


In [15]:
from sklearn.preprocessing import OrdinalEncoder

object_cols = [cname for cname in X_train.columns if X_train[cname].dtype == 'object']

label_X_train = X_train.copy()
label_X_test = X_test.copy()

ordinal_encoder = OrdinalEncoder()
label_X_train[object_cols] = ordinal_encoder.fit_transform(label_X_train[object_cols])
label_X_test[object_cols] = ordinal_encoder.transform(label_X_test[object_cols])

print('MAE с Ordinal Encoding:')
print(score_dataset(label_X_train, label_X_test, y_train, y_test))

MAE с Ordinal Encoding:
8821.413953991296


In [16]:
from sklearn.preprocessing import OneHotEncoder

OH_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
OH_cols_train = pd.DataFrame(OH_encoder.fit_transform(X_train[object_cols]))
OH_cols_test = pd.DataFrame(OH_encoder.transform(X_test[object_cols]))

OH_cols_train.index = X_train.index
OH_cols_test.index = X_test.index

num_X_train = X_train.drop(object_cols, axis=1)
num_X_test = X_test.drop(object_cols, axis=1)

OH_X_train = pd.concat([num_X_train, OH_cols_train], axis=1)
OH_X_test = pd.concat([num_X_test, OH_cols_test], axis=1)

OH_X_train.columns = OH_X_train.columns.astype(str)
OH_X_test.columns = OH_X_test.columns.astype(str)

print('MAE с One-Hot Encoding:')
print(score_dataset(OH_X_train, OH_X_test, y_train, y_test))

MAE с One-Hot Encoding:
8821.413953991296
